[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Boyu-Zhang-UOI/pml-f2026-notebooks/blob/main/03-classical-ml/05_knn_svm_tuning.ipynb)

# Distance, Margins, and a Search That Terminates

**Session 12 · no homework grades this · the project uses all of it**

Two model families that both measure distance, and the search procedure that
picks their hyperparameters. The through-line: **KNN and SVM are the two places
in this course where forgetting to scale does not produce a warning, it produces
a worse model that still runs.**

The last section is about budget. A grid search is a for-loop that somebody
forgot to count, and counting it before it starts is the difference between a
five-second experiment and one you kill after ten minutes.

In [1]:
import time

import numpy as np
import pandas as pd

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

rng = np.random.default_rng(21)

X, y = make_classification(n_samples=1500, n_features=8, n_informative=5,
                           n_redundant=1, class_sep=1.1, random_state=21)

# One feature is recorded in different units -- nothing is wrong with the data.
X[:, 0] = X[:, 0] * 900
X[:, 3] = X[:, 3] * 0.004

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0, stratify=y)

print(pd.DataFrame(X_train, columns=[f"f{i}" for i in range(8)])
      .agg(["mean", "std"]).T.round(4).to_string())

       mean       std
f0 -40.5689  911.1420
f1  -0.0146    1.4925
f2  -0.6236    2.0987
f3  -0.0024    0.0059
f4   0.5493    1.5070
f5   0.0013    1.5907
f6  -0.5154    1.7723
f7  -0.0551    0.9893


Column `f0` has a standard deviation in the hundreds; `f3`'s is a thousandth of
one. Euclidean distance adds the squares of differences, so `f0` decides
everything and `f3` is not in the room.

## Scaling is the experiment, not a formality

In [2]:
results = []
for name, model in (("KNN (k=5)", KNeighborsClassifier(5)),
                    ("SVM (RBF)", SVC(kernel="rbf", random_state=0))):
    for scaled in (False, True):
        pipe = make_pipeline(StandardScaler(), model) if scaled else model
        t0 = time.perf_counter()
        pipe.fit(X_train, y_train)
        fit_s = time.perf_counter() - t0
        results.append({"model": name, "scaled": scaled,
                        "test accuracy": pipe.score(X_test, y_test),
                        "fit seconds": round(fit_s, 3)})

print(pd.DataFrame(results).to_string(index=False))

    model  scaled  test accuracy  fit seconds
KNN (k=5)   False       0.557333        0.001
KNN (k=5)    True       0.893333        0.001
SVM (RBF)   False       0.514667        0.015
SVM (RBF)    True       0.898667        0.007


Both families lose several points of accuracy for no reason except the units the
data arrived in. Nothing raised an exception; the unscaled models are perfectly
valid objects that answer a question about a distorted space.

This is why every model in this course lives inside a `Pipeline` with its
scaler: it is not tidiness, it is the difference above.

## What `C` and `gamma` actually trade

`C` is the price of a margin violation — large `C` means "fit these points",
small `C` means "keep the boundary simple". `gamma` is the reach of one training
point in the RBF kernel — large `gamma` means each point influences only its
immediate neighbourhood, which is how an SVM overfits.

In [3]:
rows = []
for C in (0.1, 1, 10, 100):
    for gamma in ("scale", 0.01, 0.1, 1.0):
        pipe = make_pipeline(StandardScaler(), SVC(C=C, gamma=gamma, random_state=0))
        pipe.fit(X_train, y_train)
        rows.append({"C": C, "gamma": gamma,
                     "train": pipe.score(X_train, y_train),
                     "test": pipe.score(X_test, y_test),
                     "support vectors": int(pipe[-1].n_support_.sum())})

table = pd.DataFrame(rows)
print(table.to_string(index=False))

    C gamma    train     test  support vectors
  0.1 scale 0.892444 0.888000              687
  0.1  0.01 0.795556 0.789333              981
  0.1   0.1 0.884444 0.882667              692
  0.1   1.0 0.901333 0.797333             1113
  1.0 scale 0.931556 0.898667              402
  1.0  0.01 0.859556 0.853333              679
  1.0   0.1 0.923556 0.904000              405
  1.0   1.0 0.985778 0.904000              914
 10.0 scale 0.957333 0.912000              304
 10.0  0.01 0.892444 0.890667              473
 10.0   0.1 0.949333 0.912000              300
 10.0   1.0 1.000000 0.890667              880
100.0 scale 0.984000 0.888000              255
100.0  0.01 0.919111 0.912000              341
100.0   0.1 0.977778 0.904000              263
100.0   1.0 1.000000 0.890667              879


Read the `train` column against `test`. High `C` with high `gamma` memorises —
training accuracy approaches 1.0 while test accuracy falls, and the number of
support vectors climbs, because the model is keeping most of the training set to
describe its own boundary. The support-vector count is a capacity readout you
get for free.

**Note what this table is not:** those test scores were used to compare models,
which means the test set has now been consulted. This cell is a demonstration,
not a selection procedure. Everything below selects on cross-validation and
touches the test set exactly once, at the end.

## Counting a search before running it

A grid search costs `candidates x folds` fits, plus one refit. That number is
knowable in advance, and it is the number you should look at before pressing
enter.

In [4]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

grid = {
    "svc__C": [0.1, 1, 10, 100],
    "svc__gamma": ["scale", 0.01, 0.1, 1.0],
    "svc__kernel": ["rbf", "poly"],
}
folds = 5
candidates = int(np.prod([len(v) for v in grid.values()]))
print(f"candidates: {candidates}")
print(f"folds     : {folds}")
print(f"fits      : {candidates} x {folds} + 1 refit = {candidates * folds + 1}")

t0 = time.perf_counter()
single = make_pipeline(StandardScaler(), SVC(random_state=0)).fit(X_train, y_train)
one_fit = time.perf_counter() - t0
print(f"\none fit took {one_fit:.3f}s, so expect roughly "
      f"{(candidates * folds + 1) * one_fit:.1f}s")

candidates: 32
folds     : 5
fits      : 32 x 5 + 1 refit = 161

one fit took 0.006s, so expect roughly 1.0s


In [5]:
pipe = make_pipeline(StandardScaler(), SVC(random_state=0))

t0 = time.perf_counter()
search = GridSearchCV(pipe, grid, cv=folds, n_jobs=-1).fit(X_train, y_train)
grid_seconds = time.perf_counter() - t0

print(f"grid search: {grid_seconds:.1f}s wall clock, n_jobs=-1")
print(f"predicted from one fit     : {(candidates * folds + 1) * one_fit:.1f}s")
print(f"ratio                      : {grid_seconds / ((candidates * folds + 1) * one_fit):.0f}x")
print("best params:", search.best_params_)
print(f"best cross-validated accuracy: {search.best_score_:.4f}")

grid search: 87.9s wall clock, n_jobs=-1
predicted from one fit     : 1.0s
ratio                      : 84x
best params: {'svc__C': 10, 'svc__gamma': 0.1, 'svc__kernel': 'rbf'}
best cross-validated accuracy: 0.9102


**The estimate was badly wrong, and that is the lesson.** The arithmetic gives
you the number of fits; it silently assumes they all cost the same, and they do
not. A polynomial kernel at `C=100` on this data spends minutes where the
default RBF fit spent eight milliseconds, because a large `C` forces the
optimiser to keep chasing individual points.

So the rule has two halves: count the fits **and** time the most expensive
corner of the grid, not the default one. Notice which candidate the timing
implicates here — it is not the one anybody would have guessed.

In [6]:
slow = pd.DataFrame(search.cv_results_)[["param_svc__kernel", "param_svc__C",
                                          "param_svc__gamma", "mean_fit_time"]]
print(slow.sort_values("mean_fit_time", ascending=False).head(5).to_string(index=False))
print(f"\nslowest candidate is {slow.mean_fit_time.max() / slow.mean_fit_time.min():.0f}x "
      f"the fastest")

param_svc__kernel  param_svc__C param_svc__gamma  mean_fit_time
             poly         100.0              1.0      65.404712
             poly          10.0              1.0       7.701068
             poly           1.0              1.0       0.694991
             poly         100.0            scale       0.153582
             poly         100.0              0.1       0.089306

slowest candidate is 10109x the fastest


## Random search covers more ground per unit of budget

A grid spends its budget on a lattice: every value of `C` is tried against every
value of `gamma`, so with 4 x 4 you have tried only four distinct values of each.
A random search of the same size tries a different value of each parameter
every time.

In [7]:
from scipy.stats import loguniform

distributions = {
    "svc__C": loguniform(1e-2, 1e3),
    "svc__gamma": loguniform(1e-4, 1e0),
    "svc__kernel": ["rbf", "poly"],
}

t0 = time.perf_counter()
random_search = RandomizedSearchCV(pipe, distributions, n_iter=candidates, cv=folds,
                                   random_state=0, n_jobs=-1).fit(X_train, y_train)
random_seconds = time.perf_counter() - t0

print(f"same budget: {candidates} candidates, {random_seconds:.1f}s")
print("best params:", {k: (round(v, 4) if isinstance(v, float) else v)
                       for k, v in random_search.best_params_.items()})
print(f"best cross-validated accuracy: {random_search.best_score_:.4f}")
print(f"\ndistinct C values tried — grid: {len(grid['svc__C'])}, "
      f"random: {len(set(random_search.cv_results_['param_svc__C'].data))}")

same budget: 32 candidates, 5.5s
best params: {'svc__C': np.float64(0.8559), 'svc__gamma': np.float64(0.4077), 'svc__kernel': 'rbf'}
best cross-validated accuracy: 0.9182

distinct C values tried — grid: 4, random: 32


Same budget, a fraction of the wall clock — the random search never landed in
the expensive corner, which is luck rather than a property of the method.

The scores are usually close on a problem this small. The argument for random
search is about dimensions, not about this table: with six hyperparameters, a
grid of four values each is 4096 candidates, while a random search of 60 has
tried 60 distinct values of *every* parameter. When only one or two parameters
matter — which is typical — that is a far better use of the same compute.

## Overfitting the validation signal

`best_score_` is the mean cross-validated score of the winner **of a competition
you ran**. Choosing the maximum of thirty-two noisy estimates biases it upward,
and the more candidates you try, the more it is biased.

In [8]:
cv = pd.DataFrame(search.cv_results_)
best = cv.loc[cv.rank_test_score == 1].iloc[0]
print(f"winner's mean CV score : {best.mean_test_score:.4f}")
print(f"winner's fold spread   : +/- {best.std_test_score:.4f}")
print(f"candidates within one standard deviation of the winner: "
      f"{int((cv.mean_test_score >= best.mean_test_score - best.std_test_score).sum())}"
      f" of {len(cv)}")

winner's mean CV score : 0.9102
winner's fold spread   : +/- 0.0174
candidates within one standard deviation of the winner: 6 of 32


When a dozen candidates sit inside one standard deviation of the winner, "the
best hyperparameters" is a story about noise. Prefer the simplest model in that
band — smaller `C`, larger `gamma` reach, fewer support vectors — over the one
that happened to win.

The disciplined version of this is nested cross-validation: an outer loop to
estimate performance, an inner loop to choose parameters, so the reported score
never comes from the same folds that made the choice. It costs
`outer x inner x candidates` fits, which is why it is a diagram in the reading
and a locked test set here.

## Spending the test set, once

In [9]:
final = search.best_estimator_
print("chosen by cross-validation:", search.best_params_)
print(f"cross-validated estimate : {search.best_score_:.4f}")
print(f"held-out test accuracy   : {final.score(X_test, y_test):.4f}")
print(f"support vectors kept     : {int(final[-1].n_support_.sum())} "
      f"of {len(X_train)} training rows")

chosen by cross-validation: {'svc__C': 10, 'svc__gamma': 0.1, 'svc__kernel': 'rbf'}
cross-validated estimate : 0.9102
held-out test accuracy   : 0.9120
support vectors kept     : 300 of 1125 training rows


## What to take from this

- KNN and SVM measure distance, so scaling is part of the model, not part of the
  formatting. Put the scaler in the pipeline and the question never arises.
- `C` buys training accuracy at the cost of margin; `gamma` buys locality. The
  support-vector count tells you how much capacity you have spent.
- Count `candidates x folds` before you start a search. If the number is
  uncomfortable, shrink the grid rather than waiting.
- Random search beats grid search when only some parameters matter, which is
  most of the time.
- The winner of a search is the maximum of many noisy numbers. Report the spread
  and prefer the simplest model inside it.

## Where to go next

- **Reading, Session 12** — KNN's geometry, the SVM margin, and why random search
  wins in high dimensions.
- **Session 13** — trees, which need no scaling at all, and for the same reason:
  they never measure a distance.